# 3 — Model Training (T4 GPU — 16 GB VRAM Optimised)
## FireSpreadNet · Next Day Wildfire Spread

**Hardware target**: NVIDIA T4 (16 GB VRAM)  
**Protocol**: Holdout (train/val/test) — no cross-validation  
**Objective**: Best achievable Dice/IoU within T4 memory budget

### Positioning vs. other notebooks

| Aspect | Local GTX 1050 (2-4 GB) | **T4 (this notebook)** | Camber (crashed on T4) |
|--------|------------------------|------------------------|------------------------|
| Batch size | 4–8 | **12** | 16–32 |
| Training epochs | 30 | **50** | 80 |
| HP trials | 3 per model | **4 per model** | 5 per model |
| PI-CCA batch | 4 (accum=2) | **6 (accum=2, eff=12)** | 16 native |
| Test-time augmentation | No | **4-fold (rot only)** | 8-fold (rot + flip) |
| MC-Dropout uncertainty | No | **No** | Yes (30 passes) |
| U-Net base_filters | 16 | **32 (default)** | 32 |
| num_workers | 0 | **2** | 4 |
| Tuning epochs | 6 | **8** | 12 |
| Early stopping patience | 8 | **12** | 15 |

### Why Camber notebook crashed on T4

1. 8-fold TTA = 8× forward passes in memory  
2. MC-Dropout = 30 additional forward passes  
3. `num_workers=4` + `pin_memory` = extra RAM/VRAM pressure  
4. 80 epoch budget with 5 trials = long sustained load  

### Models compared

| Model | Type | Params | Notes |
|-------|------|--------|-------|
| CA | Physics baseline | 0 | Rothermel rules, no training |
| ConvLSTM | Deep Learning | ~350 K | Lightweight recurrent baseline |
| U-Net + Attention | Deep Learning | ~2.1 M | Dense segmentation |
| PI-CCA | Hybrid (ours) | ~1.5 M | Physics-informed, main contribution |

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1 — Environment Setup (T4 16 GB)
# ══════════════════════════════════════════════════════════════
import gc, json, os, sys, time, warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.amp import autocast
from torch.cuda.amp import GradScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', font_scale=1.1)
mpl.rcParams.update({'savefig.dpi': 200, 'figure.dpi': 120})
%matplotlib inline

# ── Project imports ──
sys.path.insert(0, os.path.abspath('..'))

from config import MODEL_CONFIG, TRAIN_CONFIG, SEED, MODELS_DIR, RESULTS_DIR, FIGURES_DIR
from src.data.dataset import get_dataloaders, FireSpreadDataset
from src.models.cellular_automata import CellularAutomataModel
from src.models.convlstm import ConvLSTMModel
from src.models.unet import UNetFire
from src.models.pi_cca import PIConvCellularAutomaton
from src.training.trainer import FocalLoss, DiceLoss, CombinedLoss, compute_metrics

# ── Load setup_config.json ──
_cfg_path = Path().resolve() / 'setup_config.json'
if not _cfg_path.exists():
    raise FileNotFoundError('setup_config.json not found — run 00_Setup.ipynb first')
cfg = json.load(open(_cfg_path))

PROCESSED_DIR    = Path(cfg['PROCESSED_DIR'])
FEATURE_CHANNELS = cfg['FEATURE_CHANNELS']
N_INPUT_CHANNELS = cfg['N_INPUT_CHANNELS']
CH               = cfg['CH']
norm_stats       = cfg['norm_stats']

# ── Device & random seeds ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)')
    torch.cuda.empty_cache()
else:
    vram_gb = 0
    print('CPU mode — no GPU detected')

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

print(f'Device: {device} | Seed: {SEED}')

## 3.1 Data Loading & Verification

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2 — Data Loading (T4 batch sizes)
# ══════════════════════════════════════════════════════════════

# T4: 16 GB VRAM → batch 12 is safe with AMP
BASE_BS = 12
BATCH_SIZES = {'convlstm': 12, 'unet': 12, 'pi_cca': 6, 'ca': 24}

loaders = get_dataloaders(
    PROCESSED_DIR, batch_size=BASE_BS, num_workers=2,
    augment_train=True, seed=SEED, stats=norm_stats,
)

for name, loader in loaders.items():
    print(f'{name}: {len(loader.dataset)} samples, {len(loader)} batches')

x_batch, y_batch = next(iter(loaders['train']))
print(f'\nBatch: X={x_batch.shape}, Y={y_batch.shape}')
print(f'Y unique values: {torch.unique(y_batch).tolist()}')

fire_pct = y_batch.float().mean().item() * 100
print(f'Fire pixel ratio: {fire_pct:.2f}%')
if fire_pct < 2.0:
    print('  WARNING: Severe imbalance (<2%) — focal_alpha=0.99 applied')

# Range check
print('\n── Post-normalisation range check (should be in [-5, 5]) ──')
all_ok = True
for ch_idx, ch_name in enumerate(FEATURE_CHANNELS):
    ch = x_batch[:, ch_idx]
    mn, mx = ch.min().item(), ch.max().item()
    flag = '[OUTLIER!]' if abs(mn) > 5.5 or abs(mx) > 5.5 else '[OK]'
    if abs(mn) > 5.5 or abs(mx) > 5.5:
        all_ok = False
    print(f'  {ch_name:>18s}: [{mn:+.2f}, {mx:+.2f}]  {flag}')

if all_ok:
    print('\nAll channels normalised correctly — training should be stable.')
else:
    print('\nERROR: Extreme values remain — check src/data/preprocessing.py')

## 3.2 Enhanced Training Infrastructure

Key features:
- **Mixed precision (AMP)** for ~2× speedup and ~50% VRAM saving
- **Linear warmup** followed by cosine annealing
- **Gradient accumulation** for effective larger batch sizes
- **Optimal threshold search** on validation set
- **4-fold TTA** (rotations only — memory-safe)
- **Running-sum metrics** to prevent GPU OOM
- **Aggressive memory cleanup** between epochs

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 3 — T4 Trainer: AMP + Warmup + Threshold Opt + TTA
#          Running-sum metrics, focal_alpha=0.99
# ══════════════════════════════════════════════════════════════
from torch.utils.data import WeightedRandomSampler
import gc


def compute_metrics_at_threshold(pred, target, threshold=0.5):
    with torch.no_grad():
        p_bin = (pred > threshold).float()
        t_bin = target.float()
        tp = (p_bin * t_bin).sum().item()
        fp = (p_bin * (1 - t_bin)).sum().item()
        fn = ((1 - p_bin) * t_bin).sum().item()
        tn = ((1 - p_bin) * (1 - t_bin)).sum().item()
        eps = 1e-8
        precision   = tp / (tp + fp + eps)
        recall      = tp / (tp + fn + eps)
        f1          = 2 * precision * recall / (precision + recall + eps)
        iou         = tp / (tp + fp + fn + eps)
        dice        = 2 * tp / (2 * tp + fp + fn + eps)
        accuracy    = (tp + tn) / (tp + fp + fn + tn + eps)
        specificity = tn / (tn + fp + eps)
    return {'iou': iou, 'dice': dice, 'f1': f1, 'precision': precision,
            'recall': recall, 'accuracy': accuracy, 'specificity': specificity,
            'threshold': threshold}


def find_optimal_threshold(preds, targets, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.90, 0.025)
    best_dice, best_t = 0, 0.5
    for t in thresholds:
        m = compute_metrics_at_threshold(preds, targets, float(t))
        if m['dice'] > best_dice:
            best_dice, best_t = m['dice'], float(t)
    return best_t, best_dice


def _compute_running_metrics(tp, fp, fn, tn):
    """Compute metrics from running confusion matrix sums."""
    eps = 1e-8
    precision   = tp / (tp + fp + eps)
    recall      = tp / (tp + fn + eps)
    f1          = 2 * precision * recall / (precision + recall + eps)
    iou         = tp / (tp + fp + fn + eps)
    dice        = 2 * tp / (2 * tp + fp + fn + eps)
    accuracy    = (tp + tn) / (tp + fp + fn + tn + eps)
    specificity = tn / (tn + fp + eps)
    return {'iou': iou, 'dice': dice, 'f1': f1, 'precision': precision,
            'recall': recall, 'accuracy': accuracy, 'specificity': specificity,
            'auc_pr': 0.0}


def tta_predict_light(model, x, device, use_amp=True):
    """Light TTA: 4 rotations only (no flips) → 4 passes instead of 8.
    Saves ~50% VRAM vs full TTA while still improving predictions.
    """
    preds = []
    for k in range(4):
        x_rot = torch.rot90(x, k, [2, 3]).to(device)
        if use_amp and torch.cuda.is_available():
            with autocast('cuda'):
                pred = model(x_rot)
        else:
            pred = model(x_rot)
        pred = torch.rot90(pred.float(), -k, [2, 3])
        preds.append(pred.cpu())
        del x_rot  # free GPU memory immediately
    result = torch.stack(preds).mean(dim=0)
    del preds
    return result


def make_weighted_loader(dataset, batch_size, weights, num_workers=2,
                         drop_last=True, seed=42):
    """DataLoader with pre-computed fire weights."""
    generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=len(dataset),
        replacement=True,
        generator=generator,
    )
    return DataLoader(
        dataset, batch_size=batch_size, sampler=sampler,
        num_workers=num_workers, pin_memory=True, drop_last=drop_last,
    )


class T4Trainer:
    """Training loop optimised for T4 16 GB: AMP, warmup, TTA, threshold opt.

    Uses running-sum confusion matrix (no pred accumulation during training).
    Default focal_alpha=0.99 (calibrated for ~1% fire rate).
    """

    def __init__(self, model, model_name, device, config, use_amp=True, accum_steps=1):
        self.model       = model.to(device)
        self.model_name  = model_name
        self.device      = device
        self.cfg         = config
        self.use_amp     = use_amp and torch.cuda.is_available()
        self.accum_steps = accum_steps
        self.criterion   = CombinedLoss(
            focal_w=config.get('focal_weight', 0.5),
            dice_w=config.get('dice_weight', 0.5),
            alpha=config.get('focal_alpha', 0.99),
            gamma=config.get('focal_gamma', 2.0),
        )
        params = [p for p in model.parameters() if p.requires_grad]
        self.has_params = len(params) > 0
        if self.has_params:
            self.optimizer = torch.optim.AdamW(
                params,
                lr=config.get('learning_rate', 3e-4),
                weight_decay=config.get('weight_decay', 1e-4),
            )
        else:
            self.optimizer = None
        self.scaler  = GradScaler() if self.use_amp else None
        self.history = {'train_loss': [], 'val_loss': [],
                        'train_metrics': [], 'val_metrics': [],
                        'lr': [], 'epoch_time': []}

    def _get_lr(self, epoch, total_epochs, warmup_epochs=3):
        base_lr = self.cfg.get('learning_rate', 3e-4)
        min_lr  = 1e-6
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return min_lr + 0.5 * (base_lr - min_lr) * (1 + np.cos(np.pi * progress))

    def train(self, train_loader, val_loader):
        epochs    = self.cfg.get('epochs', 50)
        patience  = self.cfg.get('early_stopping_patience', 12)
        warmup    = self.cfg.get('warmup_epochs', 4)
        grad_clip = self.cfg.get('gradient_clip', 1.0)
        best_val_dice, no_improve = 0.0, 0
        save_dir = MODELS_DIR / self.model_name
        save_dir.mkdir(parents=True, exist_ok=True)
        alpha_used = self.cfg.get('focal_alpha', 0.99)
        print(f'\n{"="*65}')
        print(f'Training {self.model_name} | {epochs} epochs | device={self.device}')
        print(f'AMP={self.use_amp} | accum_steps={self.accum_steps} | patience={patience}')
        print(f'focal_alpha={alpha_used:.3f} | focal_gamma={self.cfg.get("focal_gamma",2.0)}')
        print(f'{"="*65}')
        for epoch in range(epochs):
            t0 = time.time()
            lr = self._get_lr(epoch, epochs, warmup)
            if self.optimizer:
                for pg in self.optimizer.param_groups:
                    pg['lr'] = lr
            if self.has_params:
                train_loss, train_metrics = self._train_epoch(train_loader, grad_clip)
            else:
                train_loss, train_metrics = self._eval_epoch(val_loader)
            val_loss, val_metrics = self._eval_epoch(val_loader)
            dt = time.time() - t0
            self.history['train_loss'].append(float(train_loss))
            self.history['val_loss'].append(float(val_loss))
            self.history['train_metrics'].append(train_metrics)
            self.history['val_metrics'].append(val_metrics)
            self.history['lr'].append(float(lr))
            self.history['epoch_time'].append(float(dt))
            # Early collapse detection
            recall_flag = ' ⚠ RECALL=0 CHECK ALPHA!' if train_metrics['recall'] < 0.01 and epoch >= 2 else ''
            print(f'Epoch {epoch+1:3d}/{epochs} | '
                  f'Loss: {train_loss:.4f}/{val_loss:.4f} | '
                  f'Dice: {train_metrics["dice"]:.4f}/{val_metrics["dice"]:.4f} | '
                  f'Recall: {val_metrics["recall"]:.3f} | '
                  f'LR: {lr:.2e} | {dt:.1f}s{recall_flag}')
            if val_metrics['dice'] > best_val_dice:
                best_val_dice = val_metrics['dice']
                no_improve    = 0
                torch.save(self.model.state_dict(), save_dir / 'best_model.pt')
            else:
                no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1} (best Dice={best_val_dice:.4f})')
                break
            # Aggressive cleanup for T4
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
        with open(save_dir / 'training_history.json', 'w') as f:
            json.dump(self.history, f, indent=2, default=str)
        print(f'Best val Dice: {best_val_dice:.4f}')
        return self.history

    def _train_epoch(self, loader, grad_clip=1.0):
        """Running-sum metrics — no pred accumulation."""
        self.model.train()
        total_loss = 0
        tp_sum = fp_sum = fn_sum = tn_sum = 0
        self.optimizer.zero_grad()
        for step, (x, y) in enumerate(tqdm(loader, desc='Train', leave=False)):
            x, y = x.to(self.device, non_blocking=True), y.to(self.device, non_blocking=True)
            if self.use_amp:
                with autocast('cuda'):
                    pred = self.model(x)
                # Loss in FP32 outside autocast for stability
                loss = self.criterion(pred.float(), y.float()) / self.accum_steps
                self.scaler.scale(loss).backward()
                if (step + 1) % self.accum_steps == 0:
                    self.scaler.unscale_(self.optimizer)
                    nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()
            else:
                pred = self.model(x)
                loss = self.criterion(pred, y) / self.accum_steps
                loss.backward()
                if (step + 1) % self.accum_steps == 0:
                    nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)
                    self.optimizer.step()
                    self.optimizer.zero_grad()
            total_loss += loss.item() * self.accum_steps * x.size(0)
            with torch.no_grad():
                p_bin = (pred.detach() > 0.5).float()
                t_bin = y.float()
                tp_sum += (p_bin * t_bin).sum().item()
                fp_sum += (p_bin * (1 - t_bin)).sum().item()
                fn_sum += ((1 - p_bin) * t_bin).sum().item()
                tn_sum += ((1 - p_bin) * (1 - t_bin)).sum().item()
        avg_loss = total_loss / len(loader.dataset)
        metrics  = _compute_running_metrics(tp_sum, fp_sum, fn_sum, tn_sum)
        return avg_loss, metrics

    @torch.no_grad()
    def _eval_epoch(self, loader):
        """Running-sum metrics — no pred accumulation."""
        self.model.eval()
        total_loss = 0
        tp_sum = fp_sum = fn_sum = tn_sum = 0
        for x, y in tqdm(loader, desc='Eval', leave=False):
            x, y = x.to(self.device, non_blocking=True), y.to(self.device, non_blocking=True)
            if self.use_amp:
                with autocast('cuda'):
                    pred = self.model(x)
                loss = self.criterion(pred.float(), y.float())
            else:
                pred = self.model(x)
                loss = self.criterion(pred, y)
            total_loss += loss.item() * x.size(0)
            p_bin = (pred > 0.5).float()
            t_bin = y.float()
            tp_sum += (p_bin * t_bin).sum().item()
            fp_sum += (p_bin * (1 - t_bin)).sum().item()
            fn_sum += ((1 - p_bin) * t_bin).sum().item()
            tn_sum += ((1 - p_bin) * (1 - t_bin)).sum().item()
        avg_loss = total_loss / len(loader.dataset)
        metrics  = _compute_running_metrics(tp_sum, fp_sum, fn_sum, tn_sum)
        return avg_loss, metrics

    @torch.no_grad()
    def evaluate_with_threshold_opt(self, val_loader, test_loader=None):
        """Threshold optimisation — stores preds on CPU."""
        self.model.eval()
        val_preds, val_targets = [], []
        for x, y in val_loader:
            x = x.to(self.device, non_blocking=True)
            if self.use_amp:
                with autocast('cuda'):
                    pred = self.model(x)
            else:
                pred = self.model(x)
            val_preds.append(pred.float().cpu())
            val_targets.append(y.float())
        val_preds   = torch.cat(val_preds)
        val_targets = torch.cat(val_targets)
        best_t, best_dice = find_optimal_threshold(val_preds, val_targets)
        val_metrics = compute_metrics_at_threshold(val_preds, val_targets, best_t)
        print(f'  Optimal threshold: {best_t:.2f} (val Dice={best_dice:.4f})')
        results = {'val': val_metrics, 'optimal_threshold': best_t}
        del val_preds, val_targets
        if test_loader is not None:
            test_preds, test_targets = [], []
            for x, y in test_loader:
                x = x.to(self.device, non_blocking=True)
                if self.use_amp:
                    with autocast('cuda'):
                        pred = self.model(x)
                else:
                    pred = self.model(x)
                test_preds.append(pred.float().cpu())
                test_targets.append(y.float())
            results['test'] = compute_metrics_at_threshold(
                torch.cat(test_preds), torch.cat(test_targets), best_t)
            del test_preds, test_targets
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return results

print('T4Trainer ready (AMP, warmup, threshold opt, running-sum metrics, alpha=0.99)')

## 3.3 Model Definitions & Hyperparameter Search Space

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 4 — Model registry & HP grid (T4: 4 trials per model)
# ══════════════════════════════════════════════════════════════

MODEL_CLASSES = {
    'ca': CellularAutomataModel,
    'convlstm': ConvLSTMModel,
    'unet': UNetFire,
    'pi_cca': PIConvCellularAutomaton,
}

# T4 has 16 GB → no need to shrink U-Net (unlike GTX 1050)
# MODEL_CONFIG['unet']['base_filters'] stays at 32 (default)

print('Model parameter counts:')
for name, cls in MODEL_CLASSES.items():
    m = cls(config=MODEL_CONFIG[name])
    n_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  {name:>10s}: {n_p:>10,} params')
    del m

# ══════════════════════════════════════════════════════════════
# focal_alpha calibrated for ~1.07% fire rate
# alpha_fire × N_fire = alpha_no_fire × N_no_fire
# → alpha ≈ 0.99
# ══════════════════════════════════════════════════════════════
HP_GRID = {
    'convlstm': [
        {'learning_rate': 3e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.99,  'focal_gamma': 2.0, 'dropout': 0.15},
        {'learning_rate': 1e-4, 'weight_decay': 5e-5, 'focal_alpha': 0.98,  'focal_gamma': 2.5, 'dropout': 0.10},
        {'learning_rate': 5e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.995, 'focal_gamma': 3.0, 'dropout': 0.20},
        {'learning_rate': 2e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.97,  'focal_gamma': 2.0, 'dropout': 0.10},
    ],
    'unet': [
        {'learning_rate': 3e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.99,  'focal_gamma': 2.0, 'dropout': 0.15},
        {'learning_rate': 1e-4, 'weight_decay': 5e-5, 'focal_alpha': 0.98,  'focal_gamma': 2.5, 'dropout': 0.10},
        {'learning_rate': 5e-4, 'weight_decay': 5e-5, 'focal_alpha': 0.995, 'focal_gamma': 3.0, 'dropout': 0.20},
        {'learning_rate': 2e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.97,  'focal_gamma': 2.0, 'dropout': 0.10},
    ],
    'pi_cca': [
        {'learning_rate': 3e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.99,  'focal_gamma': 2.0, 'dropout': 0.15},
        {'learning_rate': 1e-4, 'weight_decay': 5e-5, 'focal_alpha': 0.98,  'focal_gamma': 2.5, 'dropout': 0.10},
        {'learning_rate': 2e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.995, 'focal_gamma': 3.0, 'dropout': 0.15},
        {'learning_rate': 2e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.97,  'focal_gamma': 2.0, 'dropout': 0.10},
    ],
}

TUNING_EPOCHS = 8
FINAL_EPOCHS  = 50
PATIENCE      = 12

candidate_models = ['convlstm', 'unet', 'pi_cca']

# Pre-compute fire weights ONCE
print('\nPre-computing fire weights for sampler (one-time cost)...')
def _get_fire_weights_cached(dataset):
    cache_key = id(dataset)
    if not hasattr(_get_fire_weights_cached, '_cache'):
        _get_fire_weights_cached._cache = {}
    if cache_key in _get_fire_weights_cached._cache:
        return _get_fire_weights_cached._cache[cache_key]
    if hasattr(dataset, 'targets') and hasattr(dataset, 'indices'):
        raw_targets = dataset.targets[dataset.indices]
        fire_counts = raw_targets.mean(axis=(1, 2, 3))
        print('  Used fast path (direct array access)')
    else:
        print(f'  Loading {len(dataset)} samples...')
        fire_counts = np.array([float(dataset[i][1].mean())
                                for i in tqdm(range(len(dataset)), desc='Fire count', leave=False)])
    weights = np.maximum(fire_counts, 1e-3).astype(np.float64)
    weights = weights / weights.sum()
    result = (weights, fire_counts)
    _get_fire_weights_cached._cache[cache_key] = result
    return result

weights_cached, fire_counts = _get_fire_weights_cached(loaders['train'].dataset)
n_fire_samples = (fire_counts > 0).sum()
print(f'Fire weights: {n_fire_samples}/{len(loaders["train"].dataset)} samples with fire | '
      f'mean fire density {fire_counts.mean()*100:.3f}%')

print(f'\nTraining plan: {candidate_models}')
print(f'Tuning: {TUNING_EPOCHS} epochs × {len(HP_GRID["unet"])} trials per model')
print(f'Final:  {FINAL_EPOCHS} epochs with early stopping (patience={PATIENCE})')
print(f'Batch sizes: {BATCH_SIZES}')

## 3.4 Training Loop — All Models

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 5 — Main training loop (T4 optimised)
# ══════════════════════════════════════════════════════════════

all_results = {}
tuning_summary = {}

for model_name in candidate_models:
    print(f'\n{"═"*70}')
    print(f'  MODEL: {MODEL_CONFIG[model_name]["name"]}  ({model_name})')
    print(f'{"═"*70}')

    bs = BATCH_SIZES.get(model_name, BASE_BS)
    accum = 2 if model_name == 'pi_cca' else 1  # effective bs=12 for PI-CCA

    # Fire-weighted sampler (pre-computed weights)
    train_loader = make_weighted_loader(
        loaders['train'].dataset, batch_size=bs, weights=weights_cached,
        num_workers=2, drop_last=True, seed=SEED,
    )
    val_loader = DataLoader(
        loaders['val'].dataset, batch_size=bs, shuffle=False,
        num_workers=2, pin_memory=True,
    )
    test_loader = DataLoader(
        loaders['test'].dataset, batch_size=bs, shuffle=False,
        num_workers=2, pin_memory=True,
    )

    # ── Phase 1: Hyperparameter tuning ──
    trial_results = []
    for trial_idx, hp in enumerate(HP_GRID[model_name]):
        print(f'\n  Trial {trial_idx+1}/{len(HP_GRID[model_name])}: lr={hp["learning_rate"]}, '
              f'wd={hp["weight_decay"]}, alpha={hp["focal_alpha"]}, '
              f'gamma={hp["focal_gamma"]}, dropout={hp["dropout"]}')

        mcfg = deepcopy(MODEL_CONFIG[model_name])
        mcfg['dropout'] = hp.get('dropout', mcfg.get('dropout', 0.2))
        model = MODEL_CLASSES[model_name](config=mcfg).to(device)

        tcfg = {
            'epochs': TUNING_EPOCHS,
            'learning_rate': hp['learning_rate'],
            'weight_decay': hp['weight_decay'],
            'focal_alpha': hp['focal_alpha'],
            'focal_gamma': hp['focal_gamma'],
            'focal_weight': 0.5,
            'dice_weight': 0.5,
            'early_stopping_patience': TUNING_EPOCHS,  # no early stop during tuning
            'gradient_clip': 1.0,
            'warmup_epochs': 2,
        }

        trainer = T4Trainer(
            model, model_name + '_tuning', device, tcfg,
            use_amp=True, accum_steps=accum,
        )
        hist = trainer.train(train_loader, val_loader)

        val_dices = [m['dice'] for m in hist['val_metrics']]
        best_dice = max(val_dices)
        trial_results.append({
            'hp': hp,
            'best_dice': float(best_dice),
            'last_dice': float(val_dices[-1]),
            'best_epoch': int(np.argmax(val_dices)) + 1,
        })
        print(f'    → Best val Dice: {best_dice:.4f} @ epoch {np.argmax(val_dices)+1}')

        del model, trainer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    # Select best trial
    best_trial = max(trial_results, key=lambda x: x['best_dice'])
    best_hp = best_trial['hp']
    tuning_summary[model_name] = {
        'selection_metric': 'dice',
        'best_trial': best_trial,
        'all_trials': trial_results,
    }
    print(f'\n  ★ Best config: {best_hp}')
    print(f'    Best tuning Dice: {best_trial["best_dice"]:.4f}')

    # ── Phase 2: Final training with best config ──
    print(f'\n  Final training: {FINAL_EPOCHS} epochs...')
    mcfg = deepcopy(MODEL_CONFIG[model_name])
    mcfg['dropout'] = best_hp.get('dropout', mcfg.get('dropout', 0.2))
    model = MODEL_CLASSES[model_name](config=mcfg).to(device)

    tcfg = {
        'epochs': FINAL_EPOCHS,
        'learning_rate': best_hp['learning_rate'],
        'weight_decay': best_hp['weight_decay'],
        'focal_alpha': best_hp['focal_alpha'],
        'focal_gamma': best_hp['focal_gamma'],
        'focal_weight': 0.5,
        'dice_weight': 0.5,
        'early_stopping_patience': PATIENCE,
        'gradient_clip': 1.0,
        'warmup_epochs': 4,
    }

    trainer = T4Trainer(
        model, model_name, device, tcfg,
        use_amp=True, accum_steps=accum,
    )
    history = trainer.train(train_loader, val_loader)

    # ── Phase 3: Threshold optimisation + test evaluation ──
    ckpt_path = MODELS_DIR / model_name / 'best_model.pt'
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))

    eval_results = trainer.evaluate_with_threshold_opt(val_loader, test_loader)

    all_results[model_name] = {
        'history': history,
        'tuning': tuning_summary[model_name],
        'eval': eval_results,
    }

    # Save
    save_dir = MODELS_DIR / model_name
    with open(save_dir / 'hyperparameter_tuning_t4.json', 'w') as f:
        json.dump(tuning_summary[model_name], f, indent=2, default=str)
    with open(save_dir / 'training_history_t4.json', 'w') as f:
        json.dump(history, f, indent=2, default=str)
    with open(save_dir / 'eval_results_t4.json', 'w') as f:
        json.dump(eval_results, f, indent=2, default=str)

    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# ── Physics baseline (CA) ──
print(f'\n{"═"*70}')
print('  MODEL: Cellular Automata (physics baseline — no training)')
print(f'{"═"*70}')

ca_loader_val  = DataLoader(loaders['val'].dataset,  batch_size=24, shuffle=False, num_workers=2, pin_memory=True)
ca_loader_test = DataLoader(loaders['test'].dataset, batch_size=24, shuffle=False, num_workers=2, pin_memory=True)
ca_model = CellularAutomataModel(config=MODEL_CONFIG['ca']).to(device)
ca_trainer = T4Trainer(ca_model, 'ca', device, TRAIN_CONFIG, use_amp=False)
ca_eval = ca_trainer.evaluate_with_threshold_opt(ca_loader_val, ca_loader_test)
all_results['ca'] = {'eval': ca_eval}

print('\n✅ All models trained and evaluated!')

## 3.5 Results Summary

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6 — Results table
# ══════════════════════════════════════════════════════════════

rows = []
for name in ['ca', 'convlstm', 'unet', 'pi_cca']:
    if name not in all_results:
        continue
    r = all_results[name]
    test_m = r['eval'].get('test', r['eval'].get('val', {}))
    val_m = r['eval'].get('val', {})
    rows.append({
        'Model': MODEL_CONFIG[name]['name'],
        'Type': MODEL_CONFIG[name].get('type', ''),
        'Threshold': f"{r['eval'].get('optimal_threshold', 0.5):.2f}",
        'Val Dice': f"{val_m.get('dice', 0):.4f}",
        'Val IoU': f"{val_m.get('iou', 0):.4f}",
        'Test Dice': f"{test_m.get('dice', 0):.4f}",
        'Test IoU': f"{test_m.get('iou', 0):.4f}",
        'Test F1': f"{test_m.get('f1', 0):.4f}",
        'Precision': f"{test_m.get('precision', 0):.4f}",
        'Recall': f"{test_m.get('recall', 0):.4f}",
    })

results_df = pd.DataFrame(rows)
print('\n' + '='*90)
print('  HOLDOUT TEST SET RESULTS — T4 Training')
print('='*90)
display(results_df)

# Save
results_df.to_csv(RESULTS_DIR / 'test_results_t4.csv', index=False)
with open(RESULTS_DIR / 'all_results_t4.json', 'w') as f:
    json.dump({k: v['eval'] for k, v in all_results.items()}, f, indent=2, default=str)

print(f'\nResults saved to {RESULTS_DIR}')

## 3.6 Training Curves

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 7 — Training curves
# ══════════════════════════════════════════════════════════════

colors = {'convlstm': '#2196F3', 'unet': '#4CAF50', 'pi_cca': '#FF5722'}
labels = {k: MODEL_CONFIG[k]['name'] for k in colors}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for name in candidate_models:
    if name not in all_results or 'history' not in all_results[name]:
        continue
    h = all_results[name]['history']
    c = colors[name]
    lbl = labels[name]
    epochs_range = range(1, len(h['train_loss']) + 1)

    # Loss
    axes[0, 0].plot(epochs_range, h['train_loss'], color=c, alpha=0.6, label=f'{lbl} (train)')
    axes[0, 0].plot(epochs_range, h['val_loss'], color=c, linestyle='--', label=f'{lbl} (val)')

    # Dice
    val_dice = [m['dice'] for m in h['val_metrics']]
    train_dice = [m['dice'] for m in h['train_metrics']]
    axes[0, 1].plot(epochs_range, train_dice, color=c, alpha=0.6, label=f'{lbl} (train)')
    axes[0, 1].plot(epochs_range, val_dice, color=c, linestyle='--', label=f'{lbl} (val)')

    # IoU
    val_iou = [m['iou'] for m in h['val_metrics']]
    axes[1, 0].plot(epochs_range, val_iou, color=c, label=lbl)

    # LR
    axes[1, 1].plot(epochs_range, h['lr'], color=c, label=lbl)

axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training & Validation Loss', fontweight='bold')
axes[0, 0].legend(fontsize=7)

axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Dice')
axes[0, 1].set_title('Dice Score', fontweight='bold')
axes[0, 1].legend(fontsize=7)

axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('IoU')
axes[1, 0].set_title('Validation IoU', fontweight='bold')
axes[1, 0].legend(fontsize=8)

axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('LR')
axes[1, 1].set_title('Learning Rate (Warmup + Cosine)', fontweight='bold')
axes[1, 1].legend(fontsize=8)
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves_t4.png', dpi=200, bbox_inches='tight')
plt.show()

## 3.7 Qualitative Predictions (with 4-fold TTA)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 8 — Visual predictions with light TTA
# ══════════════════════════════════════════════════════════════

x_test, y_test = next(iter(loaders['test']))
n_samples = min(4, x_test.shape[0])
n_models = len(candidate_models)

fig, axes = plt.subplots(n_samples, n_models + 2, figsize=(4*(n_models+2), 4*n_samples))
if n_samples == 1:
    axes = axes[np.newaxis, :]

for row in range(n_samples):
    fire_in = x_test[row, CH['prev_fire_mask']].numpy()
    gt = y_test[row].squeeze().numpy()

    axes[row, 0].imshow(fire_in, cmap='hot', vmin=0, vmax=max(1, fire_in.max()))
    if row == 0: axes[row, 0].set_title('Input Fire', fontweight='bold')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(gt, cmap='hot', vmin=0, vmax=1)
    if row == 0: axes[row, 1].set_title('Ground Truth', fontweight='bold')
    axes[row, 1].axis('off')

    for col, mname in enumerate(candidate_models):
        ckpt = MODELS_DIR / mname / 'best_model.pt'
        if not ckpt.exists():
            continue

        model = MODEL_CLASSES[mname](config=MODEL_CONFIG[mname]).to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
        model.eval()

        opt_t = all_results.get(mname, {}).get('eval', {}).get('optimal_threshold', 0.5)

        # Light TTA (4-fold rotations)
        with torch.no_grad():
            pred = tta_predict_light(model, x_test[row:row+1], device, use_amp=True)
        pred_np = pred.squeeze().cpu().numpy()

        axes[row, col+2].imshow(pred_np > opt_t, cmap='hot', vmin=0, vmax=1)
        if row == 0:
            axes[row, col+2].set_title(f'{MODEL_CONFIG[mname]["name"]}\n(τ={opt_t:.2f}, +TTA4)',
                                       fontweight='bold', fontsize=9)
        axes[row, col+2].axis('off')
        del model

plt.suptitle('Test Set Predictions — T4 (with 4-fold TTA)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'predictions_t4.png', dpi=200, bbox_inches='tight')
plt.show()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 3.8 Threshold Sensitivity Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 9 — Threshold sensitivity
# ══════════════════════════════════════════════════════════════

thresholds = np.arange(0.05, 0.95, 0.025)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for mname in candidate_models:
    ckpt = MODELS_DIR / mname / 'best_model.pt'
    if not ckpt.exists():
        continue

    model = MODEL_CLASSES[mname](config=MODEL_CONFIG[mname]).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    model.eval()

    val_preds, val_targets = [], []
    with torch.no_grad():
        for x, y in loaders['val']:
            x = x.to(device)
            if torch.cuda.is_available():
                with autocast('cuda'):
                    pred = model(x)
            else:
                pred = model(x)
            val_preds.append(pred.float().cpu())
            val_targets.append(y)

    val_preds = torch.cat(val_preds)
    val_targets = torch.cat(val_targets)

    dices, ious, f1s = [], [], []
    for t in thresholds:
        m = compute_metrics_at_threshold(val_preds, val_targets, float(t))
        dices.append(m['dice'])
        ious.append(m['iou'])
        f1s.append(m['f1'])

    c = colors[mname]
    axes[0].plot(thresholds, dices, color=c, marker='.', markersize=3, label=labels[mname])
    axes[1].plot(thresholds, ious, color=c, marker='.', markersize=3, label=labels[mname])
    axes[2].plot(thresholds, f1s, color=c, marker='.', markersize=3, label=labels[mname])

    del model, val_preds, val_targets
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for ax, metric in zip(axes, ['Dice', 'IoU', 'F1']):
    ax.set_xlabel('Threshold τ')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} vs Threshold (Validation)', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'threshold_sensitivity_t4.png', dpi=200, bbox_inches='tight')
plt.show()

## 3.9 Tuning Summary

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 10 — Tuning summary (HP search results)
# ══════════════════════════════════════════════════════════════

print('\n' + '='*80)
print('  HYPERPARAMETER TUNING SUMMARY')
print('='*80)

for model_name in candidate_models:
    if model_name not in tuning_summary:
        continue
    ts = tuning_summary[model_name]
    print(f'\n{MODEL_CONFIG[model_name]["name"]}:')
    print(f'  Best trial Dice: {ts["best_trial"]["best_dice"]:.4f}')
    print(f'  Best HP: {ts["best_trial"]["hp"]}')
    print(f'  All trials:')
    for i, trial in enumerate(ts['all_trials']):
        marker = ' ★' if trial == ts['best_trial'] else ''
        print(f'    [{i+1}] Dice={trial["best_dice"]:.4f} @ epoch {trial["best_epoch"]} | '
              f'lr={trial["hp"]["learning_rate"]}, wd={trial["hp"]["weight_decay"]}{marker}')

# Save
with open(RESULTS_DIR / 'tuning_summary_t4.json', 'w') as f:
    json.dump(tuning_summary, f, indent=2, default=str)

print(f'\nSaved to {RESULTS_DIR / "tuning_summary_t4.json"}')

## Summary

### T4 notebook improvements vs. Local GTX 1050

1. **Full U-Net** (base_filters=32) — 2.1M params vs 550K on GTX 1050  
2. **Larger batches** (12 vs 8) — better gradient estimates  
3. **50 epochs** (vs 30) — more time to converge  
4. **4 HP trials** (vs 3) — slightly wider search  
5. **8-epoch tuning** (vs 6) — more reliable HP selection  
6. **4-fold TTA** at inference — improved predictions at no training cost  
7. **num_workers=2** — faster data loading  
8. **patience=12** (vs 8) — avoids premature stopping  

### Why this doesn't crash on T4 (unlike Camber notebook)

1. **No 8-fold TTA** — 4-fold rotations only (half the memory)  
2. **No MC-Dropout** — the 30-pass uncertainty estimation is skipped  
3. **num_workers=2** (not 4) — less OS-level memory pressure  
4. **Batch 12** (not 16) — comfortable margin within 16 GB  
5. **50 epochs** (not 80) — shorter sustained load  
6. **Aggressive `gc.collect()` + `empty_cache()`** between trials  

### Next steps

- **Notebook 04**: Detailed test-set evaluation  
- **Notebook 05**: SHAP/Grad-CAM explainability